In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import time
import torch
import torchvision
from torchvision import datasets, transforms
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_extraction.text import TfidfVectorizer
from torchvision import datasets, transforms
import torch
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import accuracy_score
import pandas as pd
from bertopic import BERTopic
from sklearn.pipeline import Pipeline
from scipy.sparse import csr_matrix
from sklearn.model_selection import GridSearchCV
from bertopic.vectorizers import ClassTfidfTransformer
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import torch
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import torch
import torchvision
import torchvision.transforms as transforms
from collections import Counter
import numpy as np

import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

C:\Users\brush\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset Reduced

In [56]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np

def reduce_data_other():
# Define the transformation to normalize the data
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # Load the full MNIST dataset
    full_trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    full_testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    # Function to sample a reduced dataset while maintaining class distribution
    def reduce_dataset(dataset, target_size):
        labels = np.array(dataset.targets)
        unique_classes, class_counts = np.unique(labels, return_counts=True)
        
        # Compute how many samples per class
        class_ratios = class_counts / class_counts.sum()
        samples_per_class = (class_ratios * target_size).astype(int)

        indices = []
        for c, n_samples in zip(unique_classes, samples_per_class):
            class_indices = np.where(labels == c)[0]
            selected_indices = np.random.choice(class_indices, n_samples, replace=False)
            indices.extend(selected_indices)
        
        # Extract the subset of data and labels
        subset_data = dataset.data[indices]
        subset_targets = dataset.targets[indices]
        
        # Create a new MNIST dataset object with the reduced subset
        subset_dataset = torchvision.datasets.MNIST(
            root='./data', train=dataset.train, download=True, transform=transform)
        
        # Override the data and targets with the selected subset
        subset_dataset.data = subset_data
        subset_dataset.targets = subset_targets
        
        return subset_dataset

    # Reduce training and testing datasets to match the original dataset format
    reduced_trainset = reduce_dataset(full_trainset, 500)
    reduced_testset = reduce_dataset(full_testset, 100)

    # Create data loaders
    trainloader = torch.utils.data.DataLoader(reduced_trainset, batch_size=64, shuffle=True)
    testloader = torch.utils.data.DataLoader(reduced_testset, batch_size=64, shuffle=False)
    return reduced_trainset, reduced_testset, trainloader, testloader


# Reduced for NB

In [57]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np

def reduce_data_nb():
# Define the transformation to normalize the data
    transform = transforms.Compose([
        transforms.ToTensor()
    ])

    # Load the full MNIST dataset
    full_trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    full_testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    # Function to sample a reduced dataset while maintaining class distribution
    def reduce_dataset(dataset, target_size):
        labels = np.array(dataset.targets)
        unique_classes, class_counts = np.unique(labels, return_counts=True)
        
        # Compute how many samples per class
        class_ratios = class_counts / class_counts.sum()
        samples_per_class = (class_ratios * target_size).astype(int)

        indices = []
        for c, n_samples in zip(unique_classes, samples_per_class):
            class_indices = np.where(labels == c)[0]
            selected_indices = np.random.choice(class_indices, n_samples, replace=False)
            indices.extend(selected_indices)
        
        # Extract the subset of data and labels
        subset_data = dataset.data[indices]
        subset_targets = dataset.targets[indices]
        
        # Create a new MNIST dataset object with the reduced subset
        subset_dataset = torchvision.datasets.MNIST(
            root='./data', train=dataset.train, download=True, transform=transform)
        
        # Override the data and targets with the selected subset
        subset_dataset.data = subset_data
        subset_dataset.targets = subset_targets
        
        return subset_dataset

    # Reduce training and testing datasets to match the original dataset format
    reduced_trainset = reduce_dataset(full_trainset, 500)
    reduced_testset = reduce_dataset(full_testset, 100)

    # Create data loaders
    trainloader = torch.utils.data.DataLoader(reduced_trainset, batch_size=64, shuffle=True)
    testloader = torch.utils.data.DataLoader(reduced_testset, batch_size=64, shuffle=False)
    return reduced_trainset, reduced_testset, trainloader, testloader


In [58]:

# Define the 4-layer CNN model

class FourLayerCNN(nn.Module):
    def __init__(self):
        super(FourLayerCNN, self).__init__()
        # First convolutional layer
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        # Second convolutional layer
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        # Third convolutional layer
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        # Fourth convolutional layer
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        # Fully connected layers
        self.fc1 = nn.Linear(256 * 1 * 1, 512)  # Adjust input size based on the output size of conv layers
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.maxpool(x)
        x = self.relu(self.conv2(x))
        x = self.maxpool(x)
        x = self.relu(self.conv3(x))
        x = self.maxpool(x)
        x = self.relu(self.conv4(x))
        x = self.maxpool(x)
        #print(x.shape)
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the model, loss function, and optimizer
model = FourLayerCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
def train_model(model, trainloader, criterion, optimizer, epochs):
    model.train()
    for epoch in range(epochs):
        start_time = time.time()
        running_loss = 0.0
        for images, labels in trainloader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(trainloader)}")
        print("--- %s seconds ---" % (time.time() - start_time))

# Evaluate the model
def evaluate_model(model, testloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in testloader:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f'Accuracy: {100 * correct / total}%')
    return correct / total

def run_cnn(trainloader, testloader):
    train_model(model, trainloader, criterion, optimizer, epochs=16)
    accuracy = evaluate_model(model, testloader)
    return accuracy

In [59]:
class FilteredMNISTDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

def tfidf(trainloader):
    data_iter = iter(trainloader)
    images, labels = next(data_iter)

    # Convert to Numpy array for calculation
    images = images.numpy()
    labels = labels.numpy()

    # Initialize TF and IDF matrices
    pixels_per_image = images.shape[2] * images.shape[3]
    tf = np.zeros((10, pixels_per_image))  # Frequency of pixels for each digit
    idf = np.zeros(pixels_per_image)       # Global pixel frequency

    # Calculate TF and IDF
    for i in range(10):
        digit_masks = labels == i
        digit_images = images[digit_masks]
        # Calculate TF for each digit
        tf[i, :] = np.mean(digit_images.reshape(-1, pixels_per_image), axis=0)
    # Calculate IDF
    idf = np.mean(images.reshape(-1, pixels_per_image), axis=0)

    # Calculate TF-IDF
    tf_idf = np.zeros((10, pixels_per_image))
    N = 10  # Total number of digits
    for i in range(10):
        tf_idf[i, :] = tf[i, :] * np.log(N / (idf + 1e-10))  # Add a small constant to prevent division by zero

    # Set the threshold to 50% of each digit's maximum TF-IDF value
    thresholds = np.max(tf_idf, axis=1) * 0.9

    # Apply thresholds, setting pixels below the threshold to 0
    filtered_images = np.zeros_like(tf_idf)
    for i in range(10):
        mask = tf_idf[i] >= thresholds[i]
        filtered_images[i, mask] = tf_idf[i, mask]
    # Apply threshold, setting pixels below the threshold to 0 (black), and above the threshold to 1 (white)
    binary_images = np.zeros_like(tf_idf)
    for i in range(10):
        mask = tf_idf[i] >= thresholds[i]
        binary_images[i, mask] = 1  # Set pixels above the threshold to white


    # Reshape binary_images to (num_samples, 1, 28, 28) format
    filtered_images_reshaped = binary_images.reshape(10, 1, 28, 28)

    # Create labels corresponding to the digit
    labels = np.arange(10)  # Labels 0-9

    # Convert to PyTorch tensors
    filtered_images_tensor = torch.tensor(filtered_images_reshaped, dtype=torch.float32)
    labels_tensor = torch.tensor(labels, dtype=torch.long)

# Create a custom dataset


    # Define any additional transformations if needed
    transform = transforms.Compose([
        transforms.Normalize((0.5,), (0.5,))  # Normalize to match CNN expectations
    ])

    # Create dataset and dataloader
    filtered_dataset = FilteredMNISTDataset(filtered_images_tensor, labels_tensor, transform=transform)
    filtered_dataloader = DataLoader(filtered_dataset, batch_size=2, shuffle=True)
    return filtered_dataset, filtered_dataloader


In [60]:

def NB(trainloader, testloader):
    # Extract training images and labels
    train_images = []
    train_labels = []

    for images, labels in trainloader:
        images = images.view(images.size(0), -1).numpy() * 255  # Flatten and scale
        train_images.append(images)
        train_labels.append(labels.numpy())

    X_train = np.vstack(train_images)  # Stack batches into a single NumPy array
    y_train = np.hstack(train_labels)  # Flatten label list

    # Extract test images and labels
    test_images = []
    test_labels = []

    for images, labels in testloader:
        images = images.view(images.size(0), -1).numpy() * 255  # Flatten and scale
        test_images.append(images)
        test_labels.append(labels.numpy())

    X_test = np.vstack(test_images)  # Stack batches into a single NumPy array
    y_test = np.hstack(test_labels)  # Flatten label list

    # Train the Naive Bayes model
    model = MultinomialNB()
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)

    # Evaluate accuracy
    accuracy = accuracy_score(y_test, predictions)
    print(f'Accuracy: {accuracy:.2%}')
    return accuracy


## Naive Bayes w/ cTFIDF

In [61]:

def encode_images(region_size, overlap, threshold, dataset):
    def pad_image(image, region_size, overlap):
        height, width = image.shape[1], image.shape[2]
        stride = region_size - overlap
        pad_height = (stride - (height % stride)) % stride
        pad_width = (stride - (width % stride)) % stride
        padded_image = torch.nn.functional.pad(image, (0, pad_width, 0, pad_height), mode='constant', value=0)
        return padded_image

    def regions(image, size, overlap):
        padded_image = pad_image(image, size, overlap)
        stride = size - overlap
        regions = []
        for i in range(0, padded_image.shape[1] - overlap, stride):
            for j in range(0, padded_image.shape[2] - overlap, stride):
                region = padded_image[0, i:i+size, j:j+size]
                regions.append(region)
        return regions

    def regions(image, size, overlap):
        padded_image = pad_image(image, size, overlap)
        regions = []
        for i in range(0, padded_image.shape[1], size):
            for j in range(0, padded_image.shape[2], size):
                region = padded_image[0, i:i+size, j:j+size]
                regions.append(region)
        return regions

    def encode(region, threshold):
        region = torch.where(region < threshold, 0, 1)
        binary_str = ''.join(map(str, region.flatten().int().tolist()))
        return int(binary_str, 2)

    encoded_images = []
    for image, label in dataset:
        r = regions(image, region_size, overlap)
        r = [encode(region, threshold) for region in r]
        encoded_images.append([x for x in r if x != 0]) 

    return encoded_images

def create_groupeddf(encoded_images, dataset):
    text_data = [' '.join(map(str, img)) for img in encoded_images]
    df = pd.DataFrame({'Document': text_data, 'Label': dataset.targets.tolist()})
    return df.groupby('Label', as_index=False).agg({'Document': ' '.join})

def extract_ctfidf_features(groupeddf, score_threshold):
    ctfidf, features = BERTopic()._c_tf_idf(groupeddf, fit=True)
    ctfidf_array = ctfidf.toarray()

    ctfidf_features = {}
    for idx, topic in enumerate(groupeddf['Label']):
        top_indices = [i for i in range(len(features)) if ctfidf_array[idx][i] >= score_threshold]
        scaled_features = []
        for i in top_indices:
            term = features[i]
            count = max(1, int(ctfidf_array[idx][i] * 20000))
            scaled_features.extend([term] * count)
        ctfidf_features[topic] = scaled_features

    return ctfidf_features

def model_with_params(region_size, overlap, threshold, score_threshold, train_dataset, test_dataset):
    encoded_train = encode_images(region_size, overlap, threshold, train_dataset)
    groupeddf = create_groupeddf(encoded_train, train_dataset)
    ctfidf_features = extract_ctfidf_features(groupeddf, score_threshold)
    X_train = [' '.join(words) for words in ctfidf_features.values()]
    y_train = list(ctfidf_features.keys())

    X_test = [' '.join(map(str, img)) for img in encode_images(region_size, overlap, threshold, test_dataset)]
    y_test = test_dataset.targets.tolist()

    vectorizer = CountVectorizer()
    X_train_vectors = vectorizer.fit_transform(X_train)
    X_test_vectors = vectorizer.transform(X_test)

    model = MultinomialNB()
    model.fit(X_train_vectors, y_train)
    y_pred = model.predict(X_test_vectors)

    return accuracy_score(y_test, y_pred), groupeddf, ctfidf_features





# Load Data

In [62]:
trainset, testset, trainloader, testloader = reduce_data_other()
trainsetnb, testsetnb, trainloadernb, testloadernb = reduce_data_nb()
tfset, tfloader = tfidf(trainloader)
runs = 10

C:\Users\brush\AppData\Local\Temp\ipykernel_10976\357076566.py:45: RuntimeWarning: invalid value encountered in log
  tf_idf[i, :] = tf[i, :] * np.log(N / (idf + 1e-10))  # Add a small constant to prevent division by zero


# CNN

In [63]:
cnn_score = 0
for i in range(runs):
    cnn_score += run_cnn(trainloader, testloader)
cnn_score = (cnn_score/runs)

Epoch 1/16, Loss: 2.304152399301529
--- 0.665966272354126 seconds ---
Epoch 2/16, Loss: 2.22031232714653
--- 0.6571223735809326 seconds ---
Epoch 3/16, Loss: 1.692410171031952
--- 0.8744919300079346 seconds ---
Epoch 4/16, Loss: 1.0980911999940872
--- 0.6557295322418213 seconds ---
Epoch 5/16, Loss: 0.7158997133374214
--- 0.6358413696289062 seconds ---
Epoch 6/16, Loss: 0.5221405848860741
--- 0.673250675201416 seconds ---
Epoch 7/16, Loss: 0.3993573524057865
--- 0.5802733898162842 seconds ---
Epoch 8/16, Loss: 0.27630853559821844
--- 0.561220645904541 seconds ---
Epoch 9/16, Loss: 0.19168480299413204
--- 0.5626645088195801 seconds ---
Epoch 10/16, Loss: 0.12822873052209616
--- 0.5728240013122559 seconds ---
Epoch 11/16, Loss: 0.08173514576628804
--- 0.5895891189575195 seconds ---
Epoch 12/16, Loss: 0.12034706468693912
--- 0.629694938659668 seconds ---
Epoch 13/16, Loss: 0.1141494819894433
--- 0.5681667327880859 seconds ---
Epoch 14/16, Loss: 0.08266116701997817
--- 0.3664581775665283 s

# CNN w/ TFIDF

In [64]:
tfcnn_score=0
for i in range(runs):
    tfcnn_score += run_cnn(tfloader, testloader)
tfcnn_score = (tfcnn_score/runs)*100

Epoch 1/16, Loss: 4.3179802894592285
--- 0.05728960037231445 seconds ---
Epoch 2/16, Loss: 2.3312992095947265
--- 0.05017495155334473 seconds ---
Epoch 3/16, Loss: 2.316536378860474
--- 0.04256725311279297 seconds ---
Epoch 4/16, Loss: 2.31245756149292
--- 0.04468488693237305 seconds ---
Epoch 5/16, Loss: 2.3089175701141356
--- 0.042356014251708984 seconds ---
Epoch 6/16, Loss: 2.310270404815674
--- 0.04166889190673828 seconds ---
Epoch 7/16, Loss: 2.3080489158630373
--- 0.040632009506225586 seconds ---
Epoch 8/16, Loss: 2.307401657104492
--- 0.04026293754577637 seconds ---
Epoch 9/16, Loss: 2.3073492527008055
--- 0.04398059844970703 seconds ---
Epoch 10/16, Loss: 2.306776762008667
--- 0.041246652603149414 seconds ---
Epoch 11/16, Loss: 2.306383752822876
--- 0.03983640670776367 seconds ---
Epoch 12/16, Loss: 2.30691180229187
--- 0.04117441177368164 seconds ---
Epoch 13/16, Loss: 2.306029796600342
--- 0.04066157341003418 seconds ---
Epoch 14/16, Loss: 2.306226110458374
--- 0.04205775260

# NB

In [65]:
nb_score = 0
for i in range(runs):
    nb_score += NB(trainloadernb, testloadernb)
nb_score = nb_score/runs

Accuracy: 85.26%
Accuracy: 85.26%
Accuracy: 85.26%
Accuracy: 85.26%
Accuracy: 85.26%
Accuracy: 85.26%
Accuracy: 85.26%
Accuracy: 85.26%
Accuracy: 85.26%
Accuracy: 85.26%


# NB w/ cTFIDF

In [66]:
ctfnb_score = 0
for i in range(runs):
    score, gdf, ctfidf_features = model_with_params(6, 2, -.95, .00025, trainset, testset)
    ctfnb_score += score
ctfnb_score = (ctfnb_score/runs)

# Results Over N Runs

In [67]:
print(runs, " Runs")
print(f'CNN Accuracy: {cnn_score/100:.2%}')
print(f'CNN w/ TFIDF Accuracy: {tfcnn_score/100:.2%}')
print(f'NB Accuracy: {nb_score*10:.2%}')
print(f'NB w/ cTFIDF Accuracy: {ctfnb_score/100:.2%}')

10  Runs
CNN Accuracy: 0.93%
CNN w/ TFIDF Accuracy: 18.11%
NB Accuracy: 852.63%
NB w/ cTFIDF Accuracy: 0.41%


In [68]:
print(NB(trainloadernb, testloadernb))

Accuracy: 85.26%
0.8526315789473684
